<a href="https://colab.research.google.com/github/Hasnaincoder1/Flyrankrepo1/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hasnaincoder1/Flyrankrepo1/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

The question shape is "yes/no with an observed-ish proxy label" (`is_declining_label`), so per the method table I'm starting with **Logistic Regression** as the readable baseline model, then **Random Forest** to see whether the extra complexity actually earns its place. Per `training-honest-models`, simplicity is a feature — if the forest doesn't beat the readable model, the readable model wins by default, not the fancier one.


In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Grouped by `client_id`**, 75/25 train/test, via `GroupShuffleSplit`. A random row-level split would let the same client appear in both train and test, so the model could partly "recognize" a client's own baseline behavior rather than generalizing to a client it's never seen — which is exactly how this baseline would actually be used on a new client. I'm not doing a time-aware split here because this snapshot is a single trailing-90-day cross-section per page, not a multi-period panel (that's the warehouse's job, from ML-04 onward).

**Feature leakage guard:** per the data dictionary, `trend_direction` and `trend_pct` are the label's own definition and are excluded outright. I'm also excluding `impressions_last_30d` / `impressions_prev_30d` even though they're not explicitly flagged — the label is a deterministic threshold on exactly those two numbers, so including them would let the model near-perfectly reconstruct the label rather than learn a real pattern. I'm using the trailing-90-day aggregates instead (`impressions_90d`, `clicks_90d`, etc., log-transformed), which is what the repo's own reference pipeline (`scripts/ml_utils.py`) uses for the same reason.


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

for c in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    df[f"log_{c}"] = np.log1p(df[c])

NUMERIC = ["search_volume", "competition", "cpc", "word_count", "char_count",
           "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
           "days_with_impressions", "days_with_sessions", "content_age_days", "days_since_last_update",
           "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct"]
CATEGORICAL = ["competition_level", "content_type", "main_intent", "age_tier",
               "freshness_tier", "word_count_tier", "impression_tier"]

X = df[NUMERIC + CATEGORICAL]
y = df["is_declining_label"]
groups = df["client_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

train_clients = set(df.iloc[train_idx]["client_id"])
test_clients = set(df.iloc[test_idx]["client_id"])
print(f"train: {len(train_idx):,} rows, {len(train_clients)} clients")
print(f"test:  {len(test_idx):,} rows, {len(test_clients)} clients")
print(f"client overlap between train and test: {len(train_clients & test_clients)}")



FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/content_refresh_anonymized.csv'

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

One honest complication first: my ML-07 baseline rule (`declining_and_stale_with_demand`) has `is_declining_label` multiplied directly into its score — meaning any page it ever flags is, by construction, already a true positive. Evaluated at precision@50 that rule scores a trivial **1.0**, not because it's a good rule, but because it's circular by definition (it can only flag pages the label already agrees with). That's not a fair number to put next to a model. So alongside it I compute a **fair baseline**: the same staleness+demand logic with the label term removed — the part of the rule that's actually predictive rather than tautological — and compare the model against that, plus the base rate.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

test_df = df.iloc[test_idx]
base_rate = y_test.mean()

# ML-07 rule, as submitted (label-gated) -- shown for continuity, flagged as circular
orig_score = ((test_df["is_declining_label"] == 1)
              & test_df["days_since_last_update"].between(90, 180)
              & (test_df["impressions_last_30d"] > 0)).astype(int) * test_df["impressions_last_30d"]
orig_p50 = precision_at_k(orig_score.values, test_df["is_declining_label"].values, 50)

# Fair baseline: same staleness+demand logic, WITHOUT the label term
fair_score = test_df["days_since_last_update"].between(90, 180).astype(int) * test_df["impressions_90d"]
fair_p50 = precision_at_k(fair_score.values, test_df["is_declining_label"].values, 50)
fair_p100 = precision_at_k(fair_score.values, test_df["is_declining_label"].values, 100)

pre = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), NUMERIC),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]), CATEGORICAL),
])

rows = [
    {"model": "ML-07 rule (as submitted, label-gated -- circular, shown for continuity)",
     "precision@50": round(orig_p50, 3), "precision@100": None, "auc": None},
    {"model": "Fair baseline (staleness+demand, no label term)",
     "precision@50": round(fair_p50, 3), "precision@100": round(fair_p100, 3), "auc": None},
]

fitted = {}
for name, clf in [("Logistic Regression", LogisticRegression(max_iter=2000, random_state=42)),
                   ("Random Forest (depth=8, 300 trees)", RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1))]:
    pipe = Pipeline([("pre", pre), ("clf", clf)])
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    fitted[name] = (pipe, proba)
    rows.append({
        "model": name,
        "precision@50": round(precision_at_k(proba, y_test.values, 50), 3),
        "precision@100": round(precision_at_k(proba, y_test.values, 100), 3),
        "auc": round(roc_auc_score(y_test, proba), 3),
    })

rows.append({"model": "base rate (random)", "precision@50": round(base_rate, 3), "precision@100": round(base_rate, 3), "auc": 0.5})

comparison = pd.DataFrame(rows)
comparison

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

**Reading the table:** the fair baseline (0.30 precision@50) actually scores *below* the 0.52 base rate on this test slice — worse than picking randomly. That lines up with ML-07's own signal check, which called staleness "MIXED" rather than confirmed. Logistic Regression clears both the base rate and the fair baseline comfortably; Random Forest is weaker than Logistic Regression here, which is the "don't reward complexity alone" lesson landing for real — the extra flexibility isn't buying anything on this feature set and split, so the simpler model is the one worth keeping.

**What the model leans on** (permutation importance, top features): `log_impressions_90d`, `log_clicks_90d`, `content_age_days`, `impression_tier`, `log_sessions_90d`, `avg_position`. This makes sense — pages with more historical traffic and impressions give the model more signal to detect a real shift, and older pages have had more time to either hold up or fade. Nothing on that list looks suspiciously perfect (no single feature alone separates the classes), which is a mild reassurance against leakage rather than proof of its absence.

**Three concrete wrong cases below** — false positives (predicted declining, actually wasn't) and false negatives (missed a real decline).

In [ ]:
best_name = "Logistic Regression"
best_pipe, best_proba = fitted[best_name]

err_df = test_df.copy()
err_df["pred_proba"] = best_proba
err_df["pred_label"] = (best_proba >= 0.5).astype(int)

cols = ["content_id", "pred_proba", "is_declining_label", "avg_position", "ctr", "days_since_last_update", "impressions_90d"]

fp = err_df[(err_df["pred_label"] == 1) & (err_df["is_declining_label"] == 0)].sort_values("pred_proba", ascending=False).head(3)
fn = err_df[(err_df["pred_label"] == 0) & (err_df["is_declining_label"] == 1)].sort_values("pred_proba").head(3)

print("False positives -- model said declining, page actually wasn't:")
print(fp[cols].to_string(index=False))
print("\nFalse negatives -- model missed a page that WAS declining:")
print(fn[cols].to_string(index=False))
print()
print("Pattern: the false positives are mostly high-impression pages with low/zero CTR that")
print("look 'at risk' on paper but held steady -- the model over-weights visibility. The false")
print("negatives tend to sit at 0% CTR already, so there's little room left for the model to")
print("read a further drop from CTR alone; the actual decline is likely showing up somewhere")
print("this feature set doesn't fully capture, like competitor movement.")


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.